<a href="https://colab.research.google.com/github/Garima-Pachouri/Yes-Bank-Stock-Closing-Price-Prediction/blob/main/YesBank_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Yes Bank Stock Closing Price Prediction

##### **Project Type**    - EDA + Regression Machine Learning
##### **Contribution**    - Individual
##### **Team Member 1 -** Garima Pachouri

# **Project Summary -**

This project focuses on analyzing and predicting the monthly closing stock price of Yes Bank using historical stock market data. The dataset contains monthly Open, High, Low and Close prices from July 2005 onward. The target variable for this project is **Close**, because closing price is one of the most important indicators used by investors, traders and analysts to understand the final market value of a stock for a specific trading period.

The project was completed in a structured data-engineering and machine-learning workflow. First, the dataset was loaded safely with exception handling so that the notebook can run in one go without errors. Then the data was inspected using first view, shape, information, duplicate check, missing value check and unique value analysis. The Date column was converted into proper datetime format and additional time-based features such as Year, Month, Quarter, Month Number and Year-Month were created.

After understanding the dataset, detailed data wrangling was performed. Important business and analytical features were engineered, including Price Range, Average Price, Open-Close Difference, Daily Return Percentage, High-Low Percentage, Lag features, Moving Averages, Rolling Standard Deviation and Trend Direction. These features help the model understand price movement, volatility, short-term trend and previous month behavior.

The project includes extensive visualization and storytelling using 20 meaningful charts. These charts cover univariate, bivariate and multivariate analysis. The visualizations explain closing price trend, yearly behavior, volatility, relationship between OHLC variables, moving averages, returns, outliers, correlation and model-ready feature relationships. Each chart is supported with reason, insight and business impact explanation.

Three hypothesis tests were performed to statistically validate important assumptions about returns, price difference and relationship between opening and closing prices. Feature engineering and preprocessing were then completed using missing value handling, outlier treatment, feature selection, transformation, scaling and time-aware train-test splitting.

Three regression models were implemented: Linear Regression, Random Forest Regressor and Gradient Boosting Regressor. Their performance was evaluated using MAE, MSE, RMSE, R2 Score and cross-validation. Hyperparameter tuning was performed using GridSearchCV with TimeSeriesSplit where suitable. Finally, the best-performing model was selected based on lower error and better R2 score. The final model was saved using joblib and loaded again for a sanity check prediction on unseen data. Overall, this notebook is designed to be clean, well-commented, production-friendly and executable end-to-end.

# **GitHub Link -**

https://github.com/Garima-Pachouri/Yes-Bank-Stock-Closing-Price-Prediction

# **Problem Statement**

Yes Bank is a well-known private sector bank in India. Stock prices of banking companies are affected by market sentiment, financial performance, volatility and macroeconomic conditions. The objective of this project is to build a machine learning regression model that can predict the **monthly closing stock price** of Yes Bank using historical Open, High, Low and engineered time-series features.

The business problem is to understand historical price movement and create a reliable prediction model that can support analytical decision-making. The model is not meant to replace financial advice, but it can help analysts study stock behavior, identify trend patterns and estimate future closing prices from available historical information.

# **General Guidelines** : -

1. Well-structured, formatted, and commented code is used throughout the notebook.
2. Exception handling is added in dataset loading and model saving/loading.
3. The notebook is designed to run in one go without manual code changes, provided the CSV file is available in the same runtime.
4. Each chart includes reason, insight and business impact.
5. Three ML models are implemented and compared.
6. The final model is saved and loaded again for deployment sanity check.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [1]:
# Import required libraries for data handling, visualization, statistics and machine learning
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
import requests

### Dataset Loading

In [ ]:
# Load Dataset from GitHub with exception handling
github_csv_url = 'https://raw.githubusercontent.com/Garima-Pachouri/Yes-Bank-Stock-Closing-Price-Prediction/main/data_YesBank_StockPrices%285%29.csv'
local_file_path = 'data_YesBank_StockPrices(5).csv'

try:
    # Download the file from GitHub
    response = requests.get(github_csv_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(local_file_path, 'wb') as f:
        f.write(response.content)

    df = pd.read_csv(local_file_path)
    print(f'Dataset loaded successfully from GitHub URL and saved to: {local_file_path}')

except requests.exceptions.RequestException as e:
    raise RuntimeError(f'Error downloading dataset from GitHub: {e}')
except Exception as e:
    raise RuntimeError(f'Error while loading dataset: {e}')

# Keep a backup copy of original data
original_df = df.copy()

### Dataset First View

In [ ]:
# Dataset First Look
display(df.head())
display(df.tail())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Check duplicate rows
print(f'Duplicate rows in dataset: {df.duplicated().sum()}')

#### Missing Values/Null Values

In [ ]:
# Missing value count and percentage
missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': (df.isnull().mean() * 100).round(2)
})
display(missing_summary)

In [ ]:
# Visualizing missing values
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Values Heatmap')
plt.show()

### What did you know about your dataset?

The dataset contains monthly Yes Bank stock price information. It has five columns: Date, Open, High, Low and Close. There are no missing values in the provided data. The numerical columns are already in numeric format, while the Date column is initially stored as object type and needs conversion into datetime format. Since the target variable is Close price, this is a supervised regression problem. The dataset is time-based, so random shuffling must be avoided during final model splitting to reduce data leakage.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset columns
print('Columns in dataset:')
print(df.columns.tolist())

# Statistical summary
print('\nNumerical Summary:')
display(df.describe())

### Variables Description

- **Date**: Month and year of the stock record.
- **Open**: Stock price at the beginning of the month.
- **High**: Highest stock price recorded during the month.
- **Low**: Lowest stock price recorded during the month.
- **Close**: Stock price at the end of the month. This is the target variable for prediction.

### Check Unique Values for each variable.

In [ ]:
# Unique values count for each column
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')
    print(df[col].unique()[:10])
    print('-' * 60)

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Data Wrangling and Feature Engineering
processed_df = df.copy()

# Remove duplicate rows if any
processed_df = processed_df.drop_duplicates().reset_index(drop=True)

# Convert Date column into datetime format. The dataset format is like Jul-05.
processed_df['Date'] = pd.to_datetime(processed_df['Date'], format='%b-%y')
processed_df = processed_df.sort_values('Date').reset_index(drop=True)

# Basic time features
processed_df['Year'] = processed_df['Date'].dt.year
processed_df['Month'] = processed_df['Date'].dt.month
processed_df['Quarter'] = processed_df['Date'].dt.quarter
processed_df['Month_Number'] = np.arange(1, len(processed_df) + 1)
processed_df['Year_Month'] = processed_df['Date'].dt.to_period('M').astype(str)

# Price movement and volatility features
processed_df['Price_Range'] = processed_df['High'] - processed_df['Low']
processed_df['Average_Price'] = (processed_df['Open'] + processed_df['High'] + processed_df['Low'] + processed_df['Close']) / 4
processed_df['Open_Close_Diff'] = processed_df['Close'] - processed_df['Open']
processed_df['Daily_Return_%'] = ((processed_df['Close'] - processed_df['Open']) / processed_df['Open']) * 100
processed_df['High_Low_%'] = ((processed_df['High'] - processed_df['Low']) / processed_df['Low']) * 100

# Lag features use previous month information and reduce leakage risk
processed_df['Close_Lag_1'] = processed_df['Close'].shift(1)
processed_df['Close_Lag_2'] = processed_df['Close'].shift(2)
processed_df['Open_Lag_1'] = processed_df['Open'].shift(1)
processed_df['Return_Lag_1'] = processed_df['Daily_Return_%'].shift(1)

# Rolling window features based on previous records
processed_df['MA_3'] = processed_df['Close'].rolling(window=3).mean()
processed_df['MA_6'] = processed_df['Close'].rolling(window=6).mean()
processed_df['MA_12'] = processed_df['Close'].rolling(window=12).mean()
processed_df['Rolling_Std_3'] = processed_df['Close'].rolling(window=3).std()
processed_df['Rolling_Std_6'] = processed_df['Close'].rolling(window=6).std()

# Trend label only for EDA/storytelling, not used as regression target
processed_df['Trend_Direction'] = np.where(processed_df['Open_Close_Diff'] >= 0, 'Positive/Up', 'Negative/Down')

# Fill rolling/lag missing values using backfill for early months
processed_df = processed_df.bfill().ffill()

print('Data wrangling completed successfully.')
display(processed_df.head())

### What all manipulations have you done and insights you found?

The Date column was converted into datetime format and the dataset was sorted chronologically. New time-based features such as Year, Month, Quarter and Month Number were created. Price movement features such as Price Range, Average Price, Open-Close Difference, Daily Return Percentage and High-Low Percentage were added to capture volatility and monthly performance. Lag features and moving averages were also created because stock prices are time-dependent and previous month behavior can help predict the current closing price. These manipulations make the dataset more useful for both EDA and machine learning.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Closing Price Trend Over Time

In [ ]:
# Chart - 1 visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=processed_df, x='Date', y='Close')
plt.title('Chart 1: Yes Bank Closing Price Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A line chart was selected because the dataset is arranged month-wise and the main objective is to understand how the closing price changed over time. This chart is suitable for identifying long-term upward trends, downward movements, sudden fluctuations and important price phases in Yes Bank stock history.

##### 2. What is/are the insight(s) found from the chart?
The closing price does not remain constant across the complete period. It shows clear growth phases, peak phases and sharp declining phases, which means the stock has experienced major changes in market valuation over time. This also indicates that time-based patterns are important for understanding the behavior of the target variable.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight can create positive business impact because investors and analysts can identify periods of growth, decline and high risk before making decisions. However, sharp fall periods indicate negative growth and weak market sentiment, so the business should treat such periods carefully while forecasting or planning investment strategies.

#### Chart - 2: Open, High, Low and Close Trend

In [ ]:
# Chart - 2 visualization
plt.figure(figsize=(14, 6))
for col in ['Open', 'High', 'Low', 'Close']:
    sns.lineplot(data=processed_df, x='Date', y=col, label=col)
plt.title('Chart 2: OHLC Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?
A multi-line chart was used because it allows Open, High, Low and Close prices to be compared together in a single time-based view. This helps in understanding whether all price components follow a similar movement pattern or whether any component behaves differently during volatile periods.

##### 2. What is/are the insight(s) found from the chart?
Open, High, Low and Close prices generally move in the same direction, showing a strong relationship among all OHLC variables. When the stock price rises or falls, all four values usually follow the same trend, which confirms that these features contain useful information for predicting the closing price.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this has a positive modeling and business impact because related OHLC features can improve prediction accuracy. At the same time, if all prices fall together, it indicates a broader negative movement in the stock, which can warn investors about possible risk or weak market confidence.

#### Chart - 3: Distribution of Closing Price

In [ ]:
# Chart - 3 visualization
plt.figure(figsize=(10, 5))
plt.hist(processed_df['Close'], bins=30, edgecolor='black')
plt.title('Chart 3: Distribution of Closing Price')
plt.xlabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A histogram was selected to study the distribution and frequency of closing price values. It helps us understand whether the target variable is normally distributed, skewed, concentrated in a specific range or affected by extreme values.

##### 2. What is/are the insight(s) found from the chart?
The closing price distribution is not perfectly normal. Most values are concentrated in lower and middle price ranges, while very high closing prices occur only in fewer months. This shows that the dataset contains unequal price behavior across different time periods.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight helps in selecting suitable preprocessing and modeling techniques. A skewed target distribution can affect error-sensitive models, so analysts must evaluate models carefully. The presence of fewer high-value months also suggests that those periods may represent unusual market growth and should not be blindly generalized.

#### Chart - 4: Boxplot of Closing Price

In [ ]:
# Chart - 4 visualization
plt.figure(figsize=(8, 5))
sns.boxplot(y=processed_df['Close'], color='C0')
plt.title('Chart 4: Boxplot of Closing Price')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A boxplot was chosen because it clearly displays the median, spread, interquartile range and possible outliers in the closing price. This chart is useful for quickly detecting extreme stock price values that may influence model training.

##### 2. What is/are the insight(s) found from the chart?
The closing price contains some high-value observations that appear far from the majority of the data. These points may represent exceptional market phases rather than normal monthly behavior, so they need to be understood instead of removed blindly.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, outlier awareness improves business and modeling decisions because extreme values can strongly affect prediction errors. If these extreme values are linked to real market events, they provide useful risk information; however, they can also create negative impact by making the model less stable if not handled properly.

#### Chart - 5: Yearly Average Closing Price

In [ ]:
# Chart - 5 visualization
yearly_close = processed_df.groupby('Year')['Close'].mean().reset_index()
plt.figure(figsize=(14, 6))
sns.barplot(data=yearly_close, x='Year', y='Close', color='C0')
plt.xticks(rotation=45)
plt.title('Chart 5: Yearly Average Closing Price')
plt.ylabel('Average Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A yearly average bar chart was selected to compare the average closing price across different years. This chart gives a clear year-wise summary and helps identify strong, weak and transition periods in the stock's performance.

##### 2. What is/are the insight(s) found from the chart?
The average closing price varies significantly from year to year. Some years show higher average prices, while other years show noticeable decline, indicating that the stock has gone through different market cycles and valuation changes.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight helps investors compare yearly performance and identify whether the stock was improving or weakening over time. Years with declining average prices indicate negative growth and may require deeper analysis of business events, market conditions or financial risk factors.

#### Chart - 6: Monthly Average Closing Price

In [ ]:
# Chart - 6 visualization
monthly_close = processed_df.groupby('Month')['Close'].mean().reset_index()
plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_close, x='Month', y='Close', color='C0')
plt.title('Chart 6: Monthly Average Closing Price')
plt.ylabel('Average Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A monthly average chart was used to check whether closing prices show any month-wise pattern or seasonality. This helps determine whether the month feature can contribute meaningful information to the machine learning model.

##### 2. What is/are the insight(s) found from the chart?
There is some month-wise variation in average closing price, but the difference is not strong enough to prove clear seasonality. The stock appears to be more influenced by long-term trend and market conditions than by a specific calendar month.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight supports better feature selection because it shows that month alone may not be the strongest predictor. It helps the business avoid over-dependence on weak seasonal assumptions and focus more on trend, lag, volatility and price-based indicators.

#### Chart - 7: Price Range Over Time

In [ ]:
# Chart - 7 visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=processed_df, x='Date', y='Price_Range')
plt.title('Chart 7: Monthly Price Range / Volatility Over Time')
plt.xlabel('Date')
plt.ylabel('High - Low')
plt.show()

##### 1. Why did you pick the specific chart?
A line chart of price range was selected to observe how the difference between High and Low price changes over time. This is useful because price range acts as a simple measure of monthly volatility and market uncertainty.

##### 2. What is/are the insight(s) found from the chart?
The price range is not uniform across all months. It increases during certain periods, showing that the stock experienced higher volatility and larger monthly price movement during uncertain or unstable market phases.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, volatility insight is very useful for risk management. Higher price range can create trading opportunities, but it also increases uncertainty and potential loss. For a business or investor, such periods need careful forecasting and risk control before making financial decisions.

#### Chart - 8: Daily Return Percentage Distribution

In [ ]:
# Chart - 8 visualization
plt.figure(figsize=(10, 5))
plt.hist(processed_df['Daily_Return_%'], bins=30, edgecolor='black')
plt.title('Chart 8: Distribution of Monthly Return Percentage')
plt.xlabel('Return %')
plt.show()

##### 1. Why did you pick the specific chart?
A return percentage distribution chart was selected to understand the frequency of monthly gains and losses. This chart helps evaluate whether the stock usually gives small stable returns or experiences large positive and negative changes.

##### 2. What is/are the insight(s) found from the chart?
The return values are spread across both positive and negative sides, showing that Yes Bank stock had both profit-making and loss-making months. The presence of wider return values indicates that the stock has experienced uncertain and volatile performance.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight supports better risk assessment. Positive returns indicate growth opportunities, while frequent or large negative returns indicate possible negative growth and investment risk. This can help investors decide position size, timing and risk tolerance.

#### Chart - 9: Open vs Close Scatter Plot

In [ ]:
# Chart - 9 visualization
plt.figure(figsize=(8, 6))
sns.scatterplot(data=processed_df, x='Open', y='Close')
plt.title('Chart 9: Open Price vs Close Price')
plt.xlabel('Open Price')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A scatter plot was chosen to examine the relationship between opening price and closing price. Since Close is the target variable, this chart helps verify whether Open can be used as a strong predictive feature.

##### 2. What is/are the insight(s) found from the chart?
The scatter plot shows a strong positive relationship between Open and Close. In most cases, when the opening price is high, the closing price also tends to be high, which suggests that opening price contains important information about monthly closing behavior.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this creates a positive impact on model development because Open price can improve closing price prediction. However, if the gap between Open and Close becomes large in some months, it may indicate sudden market reaction, uncertainty or negative investor sentiment.

#### Chart - 10: High vs Close Scatter Plot

In [ ]:
# Chart - 10 visualization
plt.figure(figsize=(8, 6))
sns.scatterplot(data=processed_df, x='High', y='Close')
plt.title('Chart 10: High Price vs Close Price')
plt.xlabel('High Price')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
This scatter plot was selected to analyze how the highest monthly price is related to the closing price. It helps identify whether the High value can explain the upper price movement and support target prediction.

##### 2. What is/are the insight(s) found from the chart?
High price shows a strong positive relationship with Close price. This means that months with higher peak prices generally also have higher closing prices, indicating that the stock often closes in line with its monthly price level.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight improves predictive modeling because High price carries useful information about market strength. But if the High price is much greater than the Close price, it may indicate selling pressure after reaching a peak, which can be a negative signal for investors.

#### Chart - 11: Low vs Close Scatter Plot

In [ ]:
# Chart - 11 visualization
plt.figure(figsize=(8, 6))
sns.scatterplot(data=processed_df, x='Low', y='Close')
plt.title('Chart 11: Low Price vs Close Price')
plt.xlabel('Low Price')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
This scatter plot was used to study the relationship between the lowest monthly price and the closing price. The Low value is important because it represents downside movement and support level during the month.

##### 2. What is/are the insight(s) found from the chart?
Low price has a strong positive relationship with Close price. When the monthly low is higher, the closing price also tends to be higher, which shows that downside levels are closely connected with the final monthly valuation.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this helps in both prediction and risk analysis. A higher Low price can indicate stronger support and positive market confidence, while a very low value may indicate weakness, panic selling or negative pressure on the stock.

#### Chart - 12: Average Price vs Close

In [ ]:
# Chart - 12 visualization
plt.figure(figsize=(8, 6))
sns.scatterplot(data=processed_df, x='Average_Price', y='Close')
plt.title('Chart 12: Average Price vs Close Price')
plt.xlabel('Average Price')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
This chart was selected to compare the engineered Average Price feature with the closing price. Average Price combines Open, High, Low and Close information, so it helps evaluate whether this engineered feature summarizes monthly stock behavior effectively.

##### 2. What is/are the insight(s) found from the chart?
Average Price is highly aligned with Close price, showing that the engineered feature captures the overall monthly price level very well. This confirms that feature engineering has created a meaningful variable for the regression model.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this has positive impact because a well-designed engineered feature can improve model performance and interpretability. It helps the business understand overall monthly valuation instead of depending on only one raw price column.

#### Chart - 13: Moving Average Trend

In [ ]:
# Chart - 13 visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=processed_df, x='Date', y='Close', label='Close')
sns.lineplot(data=processed_df, x='Date', y='MA_3', label='3-Month MA')
sns.lineplot(data=processed_df, x='Date', y='MA_6', label='6-Month MA')
sns.lineplot(data=processed_df, x='Date', y='MA_12', label='12-Month MA')
plt.title('Chart 13: Closing Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?
A moving average trend chart was selected because moving averages smooth short-term fluctuations and make the underlying trend easier to understand. This is especially useful in stock data where monthly prices can fluctuate due to temporary market movements.

##### 2. What is/are the insight(s) found from the chart?
The moving averages follow the closing price but with smoother movement. They make long-term trend direction more visible and help identify whether the stock is in an upward, downward or unstable phase.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, moving average insights help investors and analysts make more stable decisions by reducing noise. If the closing price stays below moving averages for a long time, it may indicate negative trend or weak market confidence, which is important for risk planning.

#### Chart - 14: Correlation Heatmap

In [ ]:
# Chart - 14 visualization
numeric_cols = processed_df.select_dtypes(include=np.number).columns
plt.figure(figsize=(14, 10))
sns.heatmap(processed_df[numeric_cols].corr(), annot=False, cmap='coolwarm')
plt.title('Chart 14: Correlation Heatmap')
plt.show()

##### 1. Why did you pick the specific chart?
A correlation heatmap was selected to measure the strength and direction of relationships among numerical variables. It is useful for identifying highly related features, possible multicollinearity and variables that are strongly connected with the target variable.

##### 2. What is/are the insight(s) found from the chart?
The heatmap shows strong correlation among OHLC variables and engineered price-based features. Close is strongly related to Open, High, Low, Average Price and moving average features, which confirms that these variables are important for prediction.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight helps improve model building by selecting relevant features and understanding redundancy. Strong correlation supports prediction, but very high multicollinearity can reduce interpretability in linear models, so it must be considered during model evaluation.

#### Chart - 15: Pair Plot of Important Features

In [ ]:
# Chart - 15 visualization
pair_cols = ['Open', 'High', 'Low', 'Close', 'Price_Range', 'Average_Price']
pd.plotting.scatter_matrix(processed_df[pair_cols], figsize=(14, 14), diagonal='hist')
plt.suptitle('Chart 15: Pair Plot of Important Numerical Features', y=0.92)
plt.show()

##### 1. Why did you pick the specific chart?
A pair plot was used because it gives a multivariate view of relationships among important numerical features. It helps identify linear patterns, clusters, outliers and relationships between predictors and the target in one compact visualization.

##### 2. What is/are the insight(s) found from the chart?
The pair plot shows that price-level features have strong linear patterns with Close price. It also confirms that several predictors move together, which supports the use of regression-based models for this project.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight is helpful because it validates the modeling approach before training models. Clear feature-target relationships can improve prediction reliability, while unusual scattered points may warn about abnormal market behavior or unstable periods.

#### Chart - 16: Trend Direction Count

In [ ]:
# Chart - 16 visualization
plt.figure(figsize=(8, 5))
sns.countplot(data=processed_df, x='Trend_Direction', color='C0')
plt.title('Chart 16: Count of Positive and Negative Months')
plt.xlabel('Trend Direction')
plt.ylabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?
A count plot was selected to compare the number of months with positive and negative price movement. This chart gives a simple summary of how often the stock closed above or below its opening price.

##### 2. What is/are the insight(s) found from the chart?
The stock has both positive and negative movement months, which means the market sentiment was not one-sided throughout the dataset. This shows that the stock experienced both bullish and bearish monthly behavior.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight helps in understanding sentiment balance and risk. A higher number of positive months can indicate confidence, while negative months represent possible loss periods and should be considered during investment or risk planning.

#### Chart - 17: Rolling Volatility

In [ ]:
# Chart - 17 visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=processed_df, x='Date', y='Rolling_Std_3', label='3-Month Rolling Std')
sns.lineplot(data=processed_df, x='Date', y='Rolling_Std_6', label='6-Month Rolling Std')
plt.title('Chart 17: Rolling Volatility Over Time')
plt.xlabel('Date')
plt.ylabel('Rolling Standard Deviation')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?
A rolling volatility chart was selected to observe how risk changes over time. Rolling standard deviation is useful because it captures volatility over a moving window rather than looking at one isolated month.

##### 2. What is/are the insight(s) found from the chart?
Volatility increases sharply during unstable periods and remains lower during calmer phases. This indicates that risk is not constant in the dataset and the stock has gone through phases of high uncertainty.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this is highly useful for business risk management. High rolling volatility can create negative impact because predictions become less certain and investment risk increases. Identifying such phases helps analysts apply caution and avoid overconfident decisions.

#### Chart - 18: Open-Close Difference Over Time

In [ ]:
# Chart - 18 visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=processed_df, x='Date', y='Open_Close_Diff')
plt.axhline(0, linestyle='--')
plt.title('Chart 18: Open-Close Difference Over Time')
plt.xlabel('Date')
plt.ylabel('Close - Open')
plt.show()

##### 1. Why did you pick the specific chart?
This chart was selected to analyze the difference between opening and closing price over time. It helps identify whether the stock generally closed stronger or weaker compared to where it opened during each month.

##### 2. What is/are the insight(s) found from the chart?
The Open-Close difference moves above and below zero, which shows mixed market sentiment across months. Positive values indicate months where the stock closed above the opening level, while negative values indicate weaker closing behavior.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this insight helps investors understand monthly buying or selling pressure. Positive difference can indicate bullish strength, while negative difference may indicate selling pressure, weak confidence or possible negative market reaction.

#### Chart - 19: Quarter-wise Closing Price

In [ ]:
# Chart - 19 visualization
plt.figure(figsize=(10, 5))
sns.boxplot(data=processed_df, x='Quarter', y='Close', color='C0')
plt.title('Chart 19: Quarter-wise Closing Price Distribution')
plt.xlabel('Quarter')
plt.ylabel('Close Price')
plt.show()

##### 1. Why did you pick the specific chart?
A quarter-wise boxplot was selected to compare closing price distribution across different quarters of the year. This helps check whether quarterly seasonality or financial-year patterns have any visible effect on stock price behavior.

##### 2. What is/are the insight(s) found from the chart?
Quarter-wise differences are visible, but they are not as dominant as the long-term trend. This suggests that the stock's closing price is influenced more by broader market movement and historical price behavior than by quarter alone.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this helps decide how much importance should be given to the Quarter feature. Quarter can be used as a supporting feature, but business decisions should not depend only on quarterly patterns because long-term trend and volatility appear more important.

#### Chart - 20: Feature Relationship with Target

In [ ]:
# Chart - 20 visualization
target_corr = processed_df.select_dtypes(include=np.number).corr()['Close'].sort_values(ascending=False).drop('Close')
plt.figure(figsize=(10, 8))
sns.barplot(x=target_corr.values, y=target_corr.index, color='C0')
plt.title('Chart 20: Correlation of Features with Close Price')
plt.xlabel('Correlation with Close')
plt.ylabel('Features')
plt.show()

##### 1. Why did you pick the specific chart?
A feature-target correlation bar chart was selected to rank variables based on their relationship with the closing price. This chart supports feature selection and makes the model-building process more explainable.

##### 2. What is/are the insight(s) found from the chart?
Open, High, Low, Average Price and moving average features show strong relationships with Close. This confirms that both raw OHLC variables and engineered trend features are valuable predictors for the regression model.

##### 3. Will the gained insights help creating a positive business impact? | Are there any insights that lead to negative growth? Justify with specific reason.
Yes, this creates positive impact because it improves model transparency and helps explain why specific features were selected. It also supports better decision-making by showing which variables are most useful for predicting the closing price.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about each statement.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
**Null Hypothesis H0:** The average monthly return percentage is equal to 0.

**Alternate Hypothesis H1:** The average monthly return percentage is significantly different from 0.

#### 2. Perform an appropriate statistical test.

In [ ]:
# One-sample t-test for average monthly return percentage
returns = processed_df['Daily_Return_%'].dropna()
t_stat_1, p_value_1 = stats.ttest_1samp(returns, popmean=0)
print(f'T-statistic: {t_stat_1:.4f}')
print(f'P-value: {p_value_1:.4f}')
alpha = 0.05
if p_value_1 < alpha:
    print('Reject H0: Average monthly return is significantly different from 0.')
else:
    print('Fail to Reject H0: Average monthly return is not significantly different from 0.')

##### Which statistical test have you done to obtain P-Value?
A one-sample t-test was used.

##### Why did you choose the specific statistical test?
This test is suitable because we compare the mean of one numerical sample against a known value, which is 0% return.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
**H0:** The average closing price is the same in high-volatility and low-volatility months.

**H1:** The average closing price is different in high-volatility and low-volatility months.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Independent t-test between high volatility and low volatility months
median_range = processed_df['Price_Range'].median()
high_vol_close = processed_df.loc[processed_df['Price_Range'] > median_range, 'Close']
low_vol_close = processed_df.loc[processed_df['Price_Range'] <= median_range, 'Close']

t_stat_2, p_value_2 = stats.ttest_ind(high_vol_close, low_vol_close, equal_var=False)
print(f'T-statistic: {t_stat_2:.4f}')
print(f'P-value: {p_value_2:.4f}')
if p_value_2 < alpha:
    print('Reject H0: Average closing price differs between high and low volatility months.')
else:
    print('Fail to Reject H0: No significant difference in average closing price.')

##### Which statistical test have you done to obtain P-Value?
Welch independent two-sample t-test was used.

##### Why did you choose the specific statistical test?
It compares the mean closing price of two independent groups: high-volatility months and low-volatility months.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
**H0:** Open price and Close price have no significant linear relationship.

**H1:** Open price and Close price have a significant linear relationship.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Pearson correlation test between Open and Close prices
corr_value, p_value_3 = stats.pearsonr(processed_df['Open'], processed_df['Close'])
print(f'Pearson Correlation: {corr_value:.4f}')
print(f'P-value: {p_value_3:.4f}')
if p_value_3 < alpha:
    print('Reject H0: Open and Close prices have a significant linear relationship.')
else:
    print('Fail to Reject H0: No significant linear relationship found.')

##### Which statistical test have you done to obtain P-Value?
Pearson correlation significance test was used.

##### Why did you choose the specific statistical test?
It is suitable because both Open and Close are continuous numerical variables and we want to test their linear relationship.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Final missing value check after feature engineering
print(processed_df.isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?
The original dataset had no missing values. Lag and rolling features created initial missing values, which were handled using backfill and forward fill because the dataset is time-series based and early records need nearby historical values for model execution.

### 2. Handling Outliers

In [ ]:
# IQR based outlier capping for selected numerical features
model_df = processed_df.copy()
cap_cols = ['Open', 'High', 'Low', 'Close', 'Price_Range', 'Daily_Return_%', 'High_Low_%']
for col in cap_cols:
    q1 = model_df[col].quantile(0.25)
    q3 = model_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    model_df[col] = model_df[col].clip(lower, upper)
print('Outlier capping completed for selected numerical columns.')

##### What all outlier treatment techniques have you used and why did you use those techniques?
IQR-based capping was used instead of deleting rows because the dataset is small and time-series records should not be removed unnecessarily. Capping controls extreme values while preserving data continuity.

### 3. Categorical Encoding

In [ ]:
# Encode Trend_Direction only for optional model usage
model_df['Trend_Direction_Encoded'] = model_df['Trend_Direction'].map({'Negative/Down': 0, 'Positive/Up': 1})
print(model_df[['Trend_Direction', 'Trend_Direction_Encoded']].head())

#### What all categorical encoding techniques have you used & why did you use those techniques?
The dataset is mainly numerical. Trend Direction was encoded using binary label encoding because it has only two categories: Positive/Up and Negative/Down.

### 4. Textual Data Preprocessing | (It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

This project is not a textual/NLP dataset, so textual preprocessing steps such as lower casing, tokenization, stopword removal and vectorization are not required.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Prepare features and target
feature_cols = [
    'Open', 'High', 'Low', 'Year', 'Month', 'Quarter', 'Month_Number',
    'Price_Range', 'Average_Price', 'Open_Close_Diff', 'Daily_Return_%', 'High_Low_%',
    'Close_Lag_1', 'Close_Lag_2', 'Open_Lag_1', 'Return_Lag_1',
    'MA_3', 'MA_6', 'MA_12', 'Rolling_Std_3', 'Rolling_Std_6', 'Trend_Direction_Encoded'
]
X = model_df[feature_cols]
y = model_df['Close']
print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)

#### 2. Feature Selection

In [ ]:
# Correlation based feature importance with target
feature_target_corr = pd.concat([X, y], axis=1).corr()['Close'].drop('Close').abs().sort_values(ascending=False)
display(feature_target_corr.to_frame('Absolute Correlation with Close'))

##### What all feature selection methods have you used  and why?
Correlation analysis and model-based feature importance were used. Correlation helps identify variables that strongly move with Close price, while model-based importance helps understand practical predictive contribution.

##### Which all features you found important and why?
Open, High, Low, Average Price, lag close values and moving averages are important because they directly represent price level, trend and recent market behavior. These variables strongly explain the closing price.

### 5. Data Transformation

In [ ]:
# Check skewness to understand transformation need
skewness = X.skew().sort_values(ascending=False)
display(skewness.to_frame('Skewness'))

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?
Some numerical variables are skewed because stock prices increased and declined sharply across years. For linear model pipeline, StandardScaler is used. Power transformation can be considered, but tree models do not require strict transformation.

### 6. Data Scaling

In [ ]:
# Scaling will be applied inside model pipelines where required
numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features)]
)
print('Preprocessing pipeline created successfully.')

##### Which method have you used to scale you data and why?
StandardScaler was used for Linear Regression because linear models are sensitive to feature scale. Tree-based models like Random Forest and Gradient Boosting do not strictly need scaling.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?
Dimensionality reduction is not required because the dataset has limited features and interpretability is important. Removing dimensions using PCA may reduce business explainability.

In [ ]:
# No dimensionality reduction applied
print('Dimensionality reduction not applied because feature count is manageable and interpretability is required.')

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)
No dimensionality reduction technique was used.

### 8. Data Splitting

In [ ]:
# Time-aware train-test split without shuffling
split_index = int(len(model_df) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

##### What data splitting ratio have you used and why?
An 80:20 time-aware split was used. The first 80% records were used for training and the latest 20% records were used for testing. This is suitable for stock data because future values should not be used to train past predictions.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.
This is a regression problem, not a classification problem. Therefore class imbalance is not applicable.

In [ ]:
print('Imbalanced dataset handling is not required for this regression project.')

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)
No imbalance handling technique was used because the target variable is continuous.

## ***7. ML Model Implementation***

In [ ]:
# Helper function to evaluate regression models
def evaluate_regression_model(model, X_train, X_test, y_train, y_test, model_name):
    """Fit a model and return common regression metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = {
        'Model': model_name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2 Score': r2_score(y_test, y_pred)
    }
    return metrics, y_pred

model_results = []
predictions = {}

### ML Model - 1

In [ ]:
# ML Model - 1: Linear Regression
linear_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_metrics, linear_pred = evaluate_regression_model(
    linear_model, X_train, X_test, y_train, y_test, 'Linear Regression'
)
model_results.append(linear_metrics)
predictions['Linear Regression'] = linear_pred
pd.DataFrame([linear_metrics])

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Linear Regression was used as a baseline regression model. It tries to learn a linear relationship between stock price features and closing price. It is easy to interpret and useful for comparing with advanced models.

In [ ]:
# Evaluation metric score chart for Linear Regression
pd.DataFrame([linear_metrics]).set_index('Model')[['MAE', 'RMSE', 'R2 Score']].plot(kind='bar', figsize=(8, 5))
plt.title('Linear Regression Evaluation Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Ridge Regression tuning as regularized linear model
tscv = TimeSeriesSplit(n_splits=5)
ridge_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', Ridge())])
ridge_params = {'model__alpha': [0.01, 0.1, 1, 10, 50, 100]}
ridge_grid = GridSearchCV(ridge_pipe, ridge_params, cv=tscv, scoring='neg_root_mean_squared_error')
ridge_grid.fit(X_train, y_train)
print('Best Ridge Parameters:', ridge_grid.best_params_)
ridge_metrics, ridge_pred = evaluate_regression_model(ridge_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Tuned Ridge Regression')
pd.DataFrame([ridge_metrics])

##### Which hyperparameter optimization technique have you used and why?
GridSearchCV with TimeSeriesSplit was used because it tests multiple alpha values while respecting chronological order.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.
The tuned Ridge model is compared with baseline Linear Regression using RMSE and R2. If Ridge reduces RMSE or improves R2, it is considered an improvement.

### ML Model - 2

In [ ]:
# ML Model - 2: Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42, n_estimators=200, max_depth=8)
rf_metrics, rf_pred = evaluate_regression_model(rf_model, X_train, X_test, y_train, y_test, 'Random Forest Regressor')
model_results.append(rf_metrics)
predictions['Random Forest Regressor'] = rf_pred
pd.DataFrame([rf_metrics])

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Random Forest Regressor is an ensemble model that builds many decision trees and averages their predictions. It can capture non-linear relationships and is less sensitive to outliers than simple linear models.

In [ ]:
# Evaluation metric score chart for Random Forest
pd.DataFrame([rf_metrics]).set_index('Model')[['MAE', 'RMSE', 'R2 Score']].plot(kind='bar', figsize=(8, 5))
plt.title('Random Forest Evaluation Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Random Forest
rf_params = {
    'n_estimators': [100],
    'max_depth': [4, 8, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)
rf_grid.fit(X_train, y_train)
print('Best Random Forest Parameters:', rf_grid.best_params_)
rf_tuned_metrics, rf_tuned_pred = evaluate_regression_model(rf_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Tuned Random Forest')
model_results.append(rf_tuned_metrics)
predictions['Tuned Random Forest'] = rf_tuned_pred
pd.DataFrame([rf_tuned_metrics])

##### Which hyperparameter optimization technique have you used and why?
GridSearchCV was used to search the best tree depth, number of trees and splitting conditions. TimeSeriesSplit was used to avoid future data leakage.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.
The tuned Random Forest is compared against the default Random Forest using RMSE and R2. Improvement is accepted when RMSE decreases and R2 increases.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.
MAE shows the average absolute prediction error in stock price units. RMSE penalizes large errors more strongly, which is important in financial prediction. R2 shows how much variation in closing price is explained by the model. Lower MAE/RMSE and higher R2 create better analytical confidence.

### ML Model - 3

In [ ]:
# ML Model - 3: Gradient Boosting Regressor
gb_model = GradientBoostingRegressor(random_state=42)
gb_metrics, gb_pred = evaluate_regression_model(gb_model, X_train, X_test, y_train, y_test, 'Gradient Boosting Regressor')
model_results.append(gb_metrics)
predictions['Gradient Boosting Regressor'] = gb_pred
pd.DataFrame([gb_metrics])

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Gradient Boosting Regressor builds trees sequentially where each new tree tries to correct previous errors. It is powerful for tabular regression problems and can capture non-linear patterns.

In [ ]:
# Evaluation metric score chart for Gradient Boosting
pd.DataFrame([gb_metrics]).set_index('Model')[['MAE', 'RMSE', 'R2 Score']].plot(kind='bar', figsize=(8, 5))
plt.title('Gradient Boosting Evaluation Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Gradient Boosting
gb_params = {
    'n_estimators': [100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3],
    'subsample': [0.8, 1.0]
}
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    gb_params,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)
gb_grid.fit(X_train, y_train)
print('Best Gradient Boosting Parameters:', gb_grid.best_params_)
gb_tuned_metrics, gb_tuned_pred = evaluate_regression_model(gb_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Tuned Gradient Boosting')
model_results.append(gb_tuned_metrics)
predictions['Tuned Gradient Boosting'] = gb_tuned_pred
pd.DataFrame([gb_tuned_metrics])

##### Which hyperparameter optimization technique have you used and why?
GridSearchCV was used because it systematically checks combinations of estimators, learning rate, depth and subsampling. TimeSeriesSplit keeps validation chronological.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.
The tuned Gradient Boosting model is compared with the default model. If tuning lowers RMSE and improves R2, the tuned model is preferred.

In [ ]:
# Final model comparison
results_df = pd.DataFrame(model_results + [ridge_metrics]).drop_duplicates(subset=['Model']).sort_values('RMSE')
display(results_df)

plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x='RMSE', y='Model', color='C0')
plt.title('Final Model Comparison Based on RMSE')
plt.xlabel('RMSE')
plt.ylabel('Model')
plt.show()

### 1. Which Evaluation metrics did you consider for a positive business impact and why?
RMSE, MAE and R2 Score were considered. RMSE is most important because large stock price prediction errors can mislead financial decisions. MAE gives easy-to-understand average error, and R2 explains overall predictive strength.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

In [ ]:
# Select best model based on lowest RMSE
best_model_name = results_df.iloc[0]['Model']
print('Best model based on lowest RMSE:', best_model_name)

model_lookup = {
    'Linear Regression': linear_model,
    'Tuned Ridge Regression': ridge_grid.best_estimator_,
    'Random Forest Regressor': rf_model,
    'Tuned Random Forest': rf_grid.best_estimator_,
    'Gradient Boosting Regressor': gb_model,
    'Tuned Gradient Boosting': gb_grid.best_estimator_
}
best_model = model_lookup[best_model_name]
best_model.fit(X_train, y_train)
print('Final model trained successfully.')

The model with the lowest RMSE was selected as the final prediction model because it makes the smallest large-error mistakes on unseen recent data. For stock price prediction, reducing large errors is more important than only improving average accuracy.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Feature importance / explainability
if hasattr(best_model, 'feature_importances_'):
    importance_values = best_model.feature_importances_
elif hasattr(best_model, 'named_steps') and hasattr(best_model.named_steps.get('model'), 'coef_'):
    importance_values = np.abs(best_model.named_steps['model'].coef_)
else:
    perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
    importance_values = perm.importances_mean

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importance_values
}).sort_values('Importance', ascending=False)

display(importance_df.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', color='C0')
plt.title('Top 10 Important Features for Final Model')
plt.show()

The feature importance chart explains which variables contributed most to prediction. Price-related variables such as High, Low, Open, Average Price, moving averages and lag features usually become important because they directly represent market level and recent trend.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.

In [ ]:
# Save final model and important metadata
artifact = {
    'model': best_model,
    'feature_columns': feature_cols,
    'best_model_name': best_model_name,
    'metrics': results_df.to_dict(orient='records')
}

model_file = 'yes_bank_best_model.joblib'
try:
    joblib.dump(artifact, model_file)
    print(f'Model artifact saved successfully as {model_file}')
except Exception as e:
    raise RuntimeError(f'Error while saving model: {e}')

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.

In [ ]:
# Load saved model and perform sanity check on latest available record
try:
    loaded_artifact = joblib.load(model_file)
    loaded_model = loaded_artifact['model']
    loaded_features = loaded_artifact['feature_columns']
    sample_unseen = X_test[loaded_features].tail(1)
    sanity_prediction = loaded_model.predict(sample_unseen)[0]
    actual_value = y_test.tail(1).values[0]
    print(f'Sanity Check Prediction: {sanity_prediction:.2f}')
    print(f'Actual Close Price: {actual_value:.2f}')
except Exception as e:
    raise RuntimeError(f'Error while loading or predicting with saved model: {e}')

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

In this project, Yes Bank monthly stock price data was analyzed and used to build a machine learning regression model for predicting closing price. The dataset was clean and compact, but meaningful time-based, volatility-based and lag-based features were engineered to improve prediction quality. Detailed EDA was performed using 20 charts covering trend, distribution, volatility, correlation, moving averages and feature-target relationships.

The hypothesis testing section statistically validated important observations from EDA. Feature engineering and preprocessing were handled carefully with time-series awareness to reduce leakage. Three regression model families were implemented and tuned: Linear/Ridge Regression, Random Forest and Gradient Boosting. The final model was selected using RMSE, MAE and R2 Score, with RMSE given high priority because large prediction errors can be risky in stock price analysis.

The final model was saved in joblib format and loaded again for a sanity check, making the project deployment-ready at a basic level. Future improvements can include adding market index data, banking sector indicators, news sentiment, macroeconomic features and more advanced time-series models such as ARIMA, Prophet, LSTM or XGBoost.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !***

# Problem Statement

The objective of this project is to predict the closing stock price of Yes Bank using machine learning regression techniques based on historical stock market data.

# Why Regression?

This is a regression problem because the target variable 'Close' contains continuous numerical values.

# Limitations

- Stock prices are affected by market news and economic conditions.
- Historical data alone cannot guarantee future prediction accuracy.
- Dataset size is limited for long-term forecasting.

# Future Scope

- Integration with real-time stock APIs
- Deep learning models such as LSTM
- Sentiment analysis using financial news
- Deployment using Streamlit or Flask